# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Answer.** I use the ML-04 April-1 contract: the feature window is 2026-01-01 through 2026-03-31. Traffic is expected to be heavy-tailed, so comparisons use grouped summaries and log1p volume.

In [ ]:
import os, getpass, duckdb, pandas as pd, numpy as np
HF_TOKEN=os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HuggingFace READ token: ')
con=duckdb.connect(); con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute('SET http_retries=10'); con.execute('SET http_timeout=120'); con.execute('SET enable_http_metadata_cache=false')
FACT="read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
FEATURE_START,FEATURE_END='2026-01-01','2026-03-31'; SECOND_START,SECOND_END='2025-10-01','2025-12-31'
assert FEATURE_END < '2026-04-01' and SECOND_END < FEATURE_START
assert 'fact_content_daily_performance_sample' not in FACT
print('Source:',con.sql(f"SELECT COUNT(*) AS n,MIN(report_date) AS min_date,MAX(report_date) AS max_date FROM {FACT}").df().to_string(index=False))
print('Duplicate grains:',con.sql(f"""SELECT COUNT(*) AS n FROM (SELECT report_date,client_hash_id,content_hash_id FROM {FACT} WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}' GROUP BY 1,2,3 HAVING COUNT(*)>1)""").df().to_string(index=False))
def audit_window(a,b):
 return con.sql(f"""WITH x AS (SELECT client_hash_id,content_hash_id,SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) impressions,SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) clicks,SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_sum_position ELSE 0 END) sum_position,SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) sessions,MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) has_ga4 FROM {FACT} WHERE report_date BETWEEN DATE '{a}' AND DATE '{b}' GROUP BY 1,2) SELECT *,clicks*100.0/NULLIF(impressions,0) ctr,sum_position/NULLIF(impressions,0) avg_position,LN(1+impressions) log_impressions,impressions>=100 AND sessions>0 measurable_opportunity,CASE WHEN avg_position<=3 THEN 'top_3' WHEN avg_position<=10 THEN 'page_1' WHEN avg_position<=20 THEN 'striking' WHEN avg_position<=50 THEN 'page_3_5' WHEN avg_position>50 THEN 'deep' ELSE 'no_position_or_volume' END position_tier FROM x WHERE impressions>0""").df()
primary=audit_window(FEATURE_START,FEATURE_END); secondary=audit_window(SECOND_START,SECOND_END)
print('Rows: primary',len(primary),'secondary',len(secondary)); print(primary[['impressions','clicks','sessions','ctr','avg_position']].describe(percentiles=[.5,.9,.99]).T[['count','50%','90%','99%','max']].to_string()); print(primary.has_ga4.value_counts().rename_axis('has_ga4').reset_index(name='n').to_string(index=False))


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Answers for tests 1 and 2.** Test 1 checks position tier versus CTR. Test 2 checks whether CTR is more stable with more impressions. Every table includes `n`; buckets below 50 pages are insufficient. These are observed associations, not causal effects.

In [ ]:
MIN_BUCKET_N=50
p=primary[primary.position_tier!='no_position_or_volume'].groupby('position_tier').agg(n=('content_hash_id','size'),median_ctr=('ctr','median'),clicks=('clicks','sum'),impressions=('impressions','sum')).reset_index(); p['pooled_ctr']=p.clicks*100/p.impressions.replace(0,np.nan); p['sample_status']=np.where(p.n>=MIN_BUCKET_N,'eligible','insufficient n'); print('Test 1 - position tier and CTR:'); print(p.to_string(index=False)); e=p[p.sample_status=='eligible'].sort_values('pooled_ctr'); print('Verdict test 1:', 'CONFIRMED' if len(e)>=2 and e.pooled_ctr.is_monotonic_increasing else 'MIXED')
primary['impression_bucket']=pd.cut(primary.impressions,[0,99,999,9999,np.inf],labels=['1-99','100-999','1,000-9,999','10,000+']); v=primary.groupby('impression_bucket',observed=False).agg(n=('content_hash_id','size'),median_ctr=('ctr','median'),ctr_iqr=('ctr',lambda s:s.quantile(.75)-s.quantile(.25)),median_impressions=('impressions','median')).reset_index(); v['sample_status']=np.where(v.n>=MIN_BUCKET_N,'eligible','insufficient n'); print('Test 2 - impression volume and CTR stability:'); print(v.to_string(index=False)); e=v[v.sample_status=='eligible']; print('Verdict test 2:', 'CONFIRMED' if len(e)>=2 and e.ctr_iqr.iloc[-1]<e.ctr_iqr.iloc[0] else 'MIXED')


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Answer.** I audit the `measurable_opportunity` rule: at least 100 impressions and at least one session. The session requirement is checked against GA4 availability, and the comparison is repeated in October-December.

In [ ]:
def fs(frame,w):
 o=frame.groupby('measurable_opportunity').agg(n=('content_hash_id','size'),median_impressions=('impressions','median'),median_sessions=('sessions','median'),median_ctr=('ctr','median'),ga4_coverage=('has_ga4','mean')).reset_index(); o.insert(0,'window',w); o['sample_status']=np.where(o.n>=MIN_BUCKET_N,'eligible','insufficient n'); return o
f=pd.concat([fs(primary,'2026-01-01 to 2026-03-31'),fs(secondary,'2025-10-01 to 2025-12-31')],ignore_index=True); print('Test 3 - measurable_opportunity flag:'); print(f.to_string(index=False)); c=primary.groupby(['measurable_opportunity','has_ga4']).agg(n=('content_hash_id','size'),median_ctr=('ctr','median'),median_impressions=('impressions','median')).reset_index(); c['sample_status']=np.where(c.n>=30,'eligible','insufficient n'); print('Flag cross-cut by GA4 availability:'); print(c.to_string(index=False)); print('Verdict test 3: MIXED'); print('The volume threshold is observable, but the session requirement depends on GA4 availability. The second window is a stability check; small buckets receive no verdict.')


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Practical meaning.** Position tier and impression volume are useful review context, but raw CTR and a binary flag are not universal truth. Show impression count beside CTR and treat missing GA4 coverage as not measured, not zero engagement. The second-window rerun is a stability check, not causal proof.

In [ ]:
print('Safeguards: CTR uses clicks / impressions * 100; every table shows n; no future label, identifier, product flag, query-window feature, raw query, or client detail was used.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.